In [13]:
from pyspark.sql import SparkSession

In [14]:
spark = SparkSession.builder.appName("casestudy").master("local[*]").getOrCreate()

In [15]:
customers = spark.read.csv("./data/customers.csv", header = True)
order_items = spark.read.csv("./data/order_items.csv", header = True)
orders = spark.read.csv("./data/orders.csv", header = True)
products = spark.read.csv("./data/products.csv", header = True)
returns = spark.read.csv("./data/returns.csv", header = True)


In [16]:
customers.createOrReplaceTempView("customer")

In [17]:
total_customer = spark.sql(
    """
    select count(*) as total
    from customer
    """
)
total_customer.show()

+------+
| total|
+------+
|100000|
+------+



In [18]:
returns.createOrReplaceTempView("returns")

In [19]:
returns= spark.sql(
    """
    select count(*) as total
    from returns
    """
)
returns.show()

+------+
| total|
+------+
|100000|
+------+



In [20]:
orders.createOrReplaceTempView("orders")

In [21]:
orders = spark.sql(
    """
    select count(*) as total
    from orders
    """
)
orders.show()

+-------+
|  total|
+-------+
|1000000|
+-------+



In [32]:
order_items.createOrReplaceTempView("order_items")

In [33]:
products.createOrReplaceTempView("products")

In [34]:
products = spark.sql(
    """
    select count(*) as total
    from products
    """
)
products.show()

+-----+
|total|
+-----+
|50000|
+-----+



In [35]:
total_customers_count = total_customer.collect()[0][0]
total_orders_count = orders.collect()[0][0]
total_products_count = products.collect()[0][0]
total_returns_count = returns.collect()[0][0]
data = [(
    total_customers_count,
    total_orders_count,
    total_products_count,
    total_returns_count
)]

columns = [
    "total_customers",
    "total_orders",
    "total_products",
    "total_returns"
]

df = spark.createDataFrame(data, columns)
df.write.mode("overwrite").csv("output/q1", header=True)

df.show()

+---------------+------------+--------------+-------------+
|total_customers|total_orders|total_products|total_returns|
+---------------+------------+--------------+-------------+
|         100000|     1000000|         50000|       100000|
+---------------+------------+--------------+-------------+



In [36]:
products.createOrReplaceTempView("total_sales")

In [37]:
total_sales = spark.sql(
    """
    select distinct category, sum(unit_cost*quantity) as total_sales
    from order_items o inner join 
    products p 
    on o.product_id = p.product_id
    group by category
    """
)
total_sales.write.mode("overwrite").csv("output/q2", header=True)
total_sales.show()

[Stage 52:=============================>                            (1 + 1) / 2]

+--------------+--------------------+
|      category|         total_sales|
+--------------+--------------------+
|Home & Kitchen| 5.225150140300116E8|
|        Sports| 5.126818085300336E8|
|   Electronics|5.1316740061999464E8|
|      Clothing| 5.117638649799961E8|
|         Books|5.1482305433998823E8|
|        Beauty| 5.259687848999895E8|
|          Toys| 5.137356900100023E8|
+--------------+--------------------+



In [38]:
total_purchase = spark.sql(
    """
    select 
    c.customer_id, sum(ot.selling_price*quantity) as total_purchase
    from customer c inner join 
    orders o
    on o.customer_id = c.customer_id
    inner join order_items ot
    on o.order_id = ot.order_id
    group by c.customer_id
    order by total_purchase desc
    """
)
total_purchase.write.mode("overwrite").csv("output/q3", header=True)
total_purchase.show(10)

[Stage 79:>                                                         (0 + 2) / 2]

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      93094|181569.68000000005|
|      64560|169060.39999999997|
|      23289|          161573.8|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|151037.32000000004|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|         148281.04|
+-----------+------------------+
only showing top 10 rows



In [39]:
monthly_sales_trend = spark.sql("""
     SELECT
         MONTH(o.order_date) AS month,
         ROUND(SUM(oi.quantity * oi.selling_price), 2) AS total_sales
     FROM orders o
     JOIN order_items oi
     ON o.order_id = oi.order_id
     WHERE YEAR(o.order_date) = (SELECT MAX(YEAR(order_date)) AS yr
        FROM orders)
     GROUP BY MONTH(o.order_date)
     ORDER BY month
""")
monthly_sales_trend.write.mode("overwrite").csv("output/q4", header=True)

monthly_sales_trend.show()

+-----+--------------+
|month|   total_sales|
+-----+--------------+
|    1|4.4457777576E8|
|    2| 4.153661442E8|
|    3|4.4362824541E8|
|    4|4.2782097434E8|
|    5|4.4481061895E8|
|    6|4.3170515406E8|
|    7|4.4367051912E8|
|    8|4.4109517702E8|
|    9|4.3107152608E8|
|   10|4.4136378931E8|
|   11|4.3362336404E8|
|   12|4.4271290835E8|
+-----+--------------+



In [54]:
return_percentage = spark.sql("""
SELECT
    p.category,
    ROUND(
        COUNT(r.order_id) * 100.0 / COUNT(oi.order_id),
        2
    ) AS return_percentage
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN returns r
    ON oi.order_id = r.order_id
GROUP BY p.category
ORDER BY return_percentage DESC
""")

return_percentage.write.mode("overwrite").csv("output/q5", header=True)
return_percentage.show()

[Stage 181:============================>                            (1 + 1) / 2]

+--------------+-----------------+
|      category|return_percentage|
+--------------+-----------------+
|          Toys|            10.08|
|        Beauty|            10.03|
|        Sports|            10.02|
|         Books|            10.02|
|Home & Kitchen|            10.00|
|   Electronics|            10.00|
|      Clothing|             9.98|
+--------------+-----------------+



In [77]:
payment_mode = spark.sql(
    '''
SELECT state, payment_mode, total_orders
FROM (
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS total_orders,
        ROW_NUMBER() OVER (
            PARTITION BY c.state
            ORDER BY COUNT(*) DESC
        ) rn
    FROM customer c
    JOIN orders o
    ON c.customer_id = o.customer_id
    GROUP BY c.state, o.payment_mode
) t
WHERE rn = 1
    '''
)

payment_mode.write.mode("overwrite").csv("output/q6", header=True)
payment_mode.show()

[Stage 239:============================>                            (1 + 1) / 2]

+-----+----------------+------------+
|state|    payment_mode|total_orders|
+-----+----------------+------------+
|   CA|             UPI|       20246|
|   FL|      Debit Card|       20010|
|   GA|     Net Banking|       20041|
|   IL|Cash on Delivery|       20498|
|   MI|      Debit Card|       20416|
|   NC|     Net Banking|       19596|
|   NY|      Debit Card|       20369|
|   OH|     Net Banking|       20351|
|   TX|             UPI|       20065|
|   WA|             UPI|       20244|
+-----+----------------+------------+



In [83]:
purchased_product = spark.sql(
    '''
    SELECT c.customer_id, c.customer_name,
    COUNT(DISTINCT p.category) AS category_count,
    SUM(oi.quantity * oi.selling_price) AS total_spent
    FROM customer c
    JOIN orders o
    ON c.customer_id = o.customer_id
    JOIN order_items oi
    ON o.order_id = oi.order_id
    JOIN products p
    ON oi.product_id = p.product_id
    GROUP BY c.customer_id, c.customer_name
    HAVING COUNT(DISTINCT p.category) >= 5
    AND SUM(oi.quantity * oi.selling_price) > 100000
    ORDER BY total_spent DESC;
    '''
)

purchased_product.write.mode("overwrite").csv("output/q7", header=True)
purchased_product.show()

26/06/16 06:17:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 06:17:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 298:============================>                        

+-----------+--------------+--------------+------------------+
|customer_id| customer_name|category_count|       total_spent|
+-----------+--------------+--------------+------------------+
|      93094|Customer_93094|             7|         181569.68|
|      64560|Customer_64560|             7|          169060.4|
|      23289|Customer_23289|             7|          161573.8|
|      52275|Customer_52275|             7|153364.78999999998|
|      61218|Customer_61218|             7|         153067.55|
|      52034|Customer_52034|             7|         152680.05|
|      40442|Customer_40442|             7|         151037.32|
|      60528|Customer_60528|             7|         148691.95|
|      84830|Customer_84830|             7|         148363.84|
|      82593|Customer_82593|             7|         148281.04|
|      20648|Customer_20648|             7|         148084.29|
|      15759|Customer_15759|             7|         147852.55|
|      36102|Customer_36102|             7|147833.90000